# 01b - Data Version Control: Service Orders (Tabular)

**Azure ML MLOps Workshop | Track B - Tabular Data**

### Context

In Notebook 01, you versioned the **inspection comments** dataset (text data).
Now you'll apply the same MLOps patterns to the **service orders** dataset
(structured/tabular data) — demonstrating that data versioning is data-agnostic.

### What you'll do:
1. Profile the raw service orders dataset
2. Clean the data (handle nulls, engineer date features, encode categoricals)
3. Register the cleaned version as `service-orders` v2
4. Compare v1 (raw) and v2 (cleaned) side by side

### Key difference from Track A:
- **Track A** cleaning: HTML removal, text normalization, empty-comment filtering
- **Track B** cleaning: null-row removal, date parsing, numeric conversion, categorical encoding
- **Same MLOps pattern:** immutable versions, lineage tags, reproducibility

---

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "<YOUR_SUBSCRIPTION_ID>"  # <<<< CHANGE THIS TO YOUR AZURE SUBSCRIPTION ID
RESOURCE_GROUP = "<YOUR_RESOURCE_GROUP>"  # <<<< CHANGE THIS TO YOUR RESOURCE GROUP (e.g., rg-aml-workshop-jd)
WORKSPACE_NAME = "<YOUR_WORKSPACE_NAME>"  # <<<< CHANGE THIS TO YOUR WORKSPACE NAME (e.g., aml-workshop-jd)

ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
print(f"Connected to: {ml_client.workspace_name}")

## Step 1: Profile the Raw Data

The service orders dataset was registered as `service-orders` v1 in Notebook 01.
Let's understand what we're working with before cleaning.

In [ ]:
import pandas as pd

df_raw = pd.read_csv("../../data/service_orders_dataset.csv")

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"\nColumns: {list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nNull counts:\n{df_raw.isnull().sum()}")
print(f"\nNull percentage: {df_raw.isnull().mean().mean():.1%} overall")

In [ ]:
# Target variable distribution
print("Target: RepairType")
print(df_raw["RepairType"].value_counts())
print(f"\nNull target rows: {df_raw['RepairType'].isnull().sum():,}")

print("\n--- Key feature distributions ---")
print(f"\nTop 10 equipment models:")
print(df_raw["EquipmentModel"].value_counts().head(10))

print(f"\nJob control types:")
print(df_raw["JobCode"].value_counts().head(10))

print(f"\nQtyOrdered stats:")
print(df_raw["QtyOrdered"].describe())

### Observations

Notice the ~97K rows with null values across all columns — these are empty rows
that need to be removed. This is a very different cleaning challenge compared to
the text data in Track A, where the issue was HTML tags and whitespace.

---

## Step 2: Clean and Engineer Features

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../src/track_b_tabular"))

from preprocess_os import load_and_clean_os

df_clean = load_and_clean_os("../../data/service_orders_dataset.csv")

print(f"Raw rows:     {len(df_raw):,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Rows dropped: {len(df_raw) - len(df_clean):,} ({(len(df_raw) - len(df_clean)) / len(df_raw):.1%})")
print(f"\nNew columns added: month, quarter, day_of_week, label")
print(f"\nLabel distribution:")
print(df_clean["label"].value_counts().rename({0: "Preventive", 1: "Overhaul"}))
print(f"\nDate features sample:")
print(df_clean[["OrderRequestDate", "month", "quarter", "day_of_week"]].head(10))

In [ ]:
# Save cleaned version
cleaned_path = "../../data/service_orders_cleaned.csv"
df_clean.to_csv(cleaned_path, index=False, encoding="utf-8")
print(f"Saved cleaned data to {cleaned_path}")
print(f"  Rows: {len(df_clean):,}")
print(f"  Columns: {df_clean.shape[1]}")

## Step 3: Register Cleaned Data as Version 2

Same pattern as Track A — every transformation gets its own version.
The raw data (v1) remains immutable.

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

os_data_v2 = Data(
    name="service-orders",
    version="2",
    description=(
        "Cleaned Contoso service orders - nulls removed, date features engineered, "
        "numeric label added. Ready for ML training."
    ),
    path=cleaned_path,
    type=AssetTypes.URI_FILE,
    tags={
        "source": "contoso",
        "stage": "cleaned",
        "region": "region-1",
        "rows": str(len(df_clean)),
        "parent_version": "1",
        "preprocessing": "null_removal,date_engineering,label_encoding",
        "target": "RepairType",
    },
)

os_data_v2 = ml_client.data.create_or_update(os_data_v2)
print(f"Registered: {os_data_v2.name} v{os_data_v2.version}")
print(f"  Tags: {os_data_v2.tags}")

## Step 4: Compare Versions

Query both versions programmatically — the same approach you used in Track A.

In [ ]:
print("All versions of 'service-orders':")
for version in ml_client.data.list(name="service-orders"):
    print(f"  v{version.version} | {version.description[:70]}... | Stage: {version.tags.get('stage', 'N/A')}")

print()

data_v1 = ml_client.data.get(name="service-orders", version="1")
data_v2 = ml_client.data.get(name="service-orders", version="2")

print(f"v1 rows: {data_v1.tags.get('rows')} | Stage: {data_v1.tags.get('stage')}")
print(f"v2 rows: {data_v2.tags.get('rows')} | Stage: {data_v2.tags.get('stage')}")
print(f"\nv2 derived from v1 (tag: parent_version={data_v2.tags.get('parent_version')})")

## Go to Azure ML Studio Now

Navigate to **Data > service-orders** and compare both versions side by side.
Notice how the same versioning pattern works identically for tabular data.

### Track A vs Track B Comparison

| Aspect | Track A (Inspections) | Track B (Service Orders) |
|--------|----------------------|-------------------------|
| **Data type** | Text (inspection comments) | Tabular (structured records) |
| **Raw rows** | 10,500 | 425,745 |
| **Cleaning needed** | HTML tags, whitespace, empty text | Null rows, date parsing, numeric conversion |
| **Versioning pattern** | Identical | Identical |
| **Lineage tracking** | `parent_version` tag | `parent_version` tag |

**The MLOps infrastructure doesn't care what your data looks like.**

---

**Next:** [02b - Experiment Tracking (Tabular)](./02b_experiment_tracking_tabular.ipynb)